# SkillNet AI SDK Usage Tutorial

This notebook demonstrates the process of using the `skillnet-ai` Python SDK in a general scenario.
Key features include:
1. **Search**: Find skills using keyword or semantic search
2. **Download**: Install skills from GitHub
3. **Create**: Create new skills from conversation logs, documents, or prompts
4. **Evaluate**: Evaluate skill quality across multiple dimensions
5. **Analyze**: Analyze skill relationships in the local skill library

## 0. Installation and Environment Configuration

In [ ]:
# Install skillnet-ai
%pip install skillnet-ai

In [ ]:
import os
import sys
from skillnet_ai import SkillNetClient

# Set API KEY (used for Create, Evaluate, and Analyze functions)
# Please replace with your actual API Key (e.g., OpenAI key)
os.environ["API_KEY"] = "your-api-key-here" 
os.environ["BASE_URL"] = "https://api.openai.com/v1" # Optional: Set custom LLM Base URL

# Initialize Client
# Note: Without an API Key, some functions (like Create, Evaluate) may not work, but Search usually works
client = SkillNetClient(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL")
    # github_token="ghp-..." # Optional: For accessing private repos or increasing download rate limits
)

print("SkillNetClient initialized successfully.")

SkillNetClient initialized successfully.


## 1. Search Skills (Search)

SkillNet supports two search modes:
*   **Keyword**: Traditional fuzzy keyword matching
*   **Vector**: Semantic vector retrieval

In [20]:
# 1. Standard Keywords Match
print("--- Keyword Search: 'pdf' ---")
results = client.search(q="pdf")

if results:
    for skill in results:
        print(f"Name: {skill.skill_name} (Stars: {skill.stars})")

# 2. Semantic Search
print("\n--- Vector Search: 'Help me analyze financial PDF reports' ---")
results = client.search(q="Help me analyze financial PDF reports", mode="vector")

if results:
    top_skill = results[0]
    print(f"Found: {top_skill.skill_name} (Stars: {top_skill.stars})")
    print(f"URL: {top_skill.skill_url}")

--- Keyword Search: 'pdf' ---
Name: biorxiv-database (Stars: 17616)
Name: treatment-plans (Stars: 16346)
Name: clinical-decision-support (Stars: 16091)
Name: scientific-visualization (Stars: 16082)
Name: literature-review (Stars: 16059)
Name: pdf (Stars: 15845)
Name: plotly (Stars: 15805)
Name: pdf-processing-pro (Stars: 15803)
Name: pdf-processing (Stars: 15439)
Name: pdf (Stars: 13720)
Name: pdf-processing (Stars: 9170)
Name: pdf-extractor (Stars: 8003)
Name: biorxiv-database (Stars: 6637)
Name: treatment-plans (Stars: 6170)
Name: clinical-decision-support (Stars: 6069)
Name: literature-review (Stars: 6041)
Name: etetoolkit (Stars: 5879)
Name: paper-2-web (Stars: 5867)
Name: pptx-posters (Stars: 5810)
Name: markitdown (Stars: 5761)

--- Vector Search: 'Help me analyze financial PDF reports' ---
Found: financial-document-parser (Stars: 95)
URL: https://github.com/Microck/ordinary-claude-skills/blob/76c86cb21a36c0cc10c213d991aa1c9498b363c8/skills_categorized/payment/financial-document-

## 2. Download Skills (Download)

You can download skills directly from a GitHub URL to a local directory.

In [21]:
# Define download target directory
download_dir = "./my_downloaded_skills"

# Example: Download anthropics' skill-creator skill
# Note: This requires network access to GitHub
skill_url = "https://github.com/anthropics/skills/tree/main/skills/skill-creator"

try:
    print(f"Downloading skill from {skill_url}...")
    local_path = client.download(url=skill_url, target_dir=download_dir)
    print(f"✅ Skill successfully installed at: {local_path}")
    
    # View downloaded files
    if os.path.exists(local_path):
        print("Files in directory:", os.listdir(local_path))
except Exception as e:
    print(f"❌ Download failed: {e}")
    # Note: If network is unreachable, this step may fail, but won't affect later demonstrations

✅ Skill successfully installed at: e:\科研\SkillNet相关\SkillNet\skillnet-ai\examples\my_downloaded_skills\skill-creator
Files in directory: ['LICENSE.txt', 'references', 'scripts', 'SKILL.md']


## 3. Create Skills (Create)

Automatically generate standardized skill packages using LLMs. Supports various sources:
*   Conversation Trajectory/Logs
*   GitHub Repository
*   Office Documents (PDF/Word/PPT)
*   Natural Language Prompt

In [25]:
# Set output directory for generated skills
created_skills_dir = "./my_created_skills"

# Example 1: Create from Conversation Log (Trajectory)
trajectory_log = """
User: I need to rename all .jpg files in this folder to .png.
Agent: I will write a python script to iterate through the folder...
Agent: Script executed. Renamed 5 files.
"""

print("--- Creating skill from trajectory ---")
try:
    # This step requires a valid API_KEY
    created_paths = client.create(
        trajectory_content=trajectory_log, 
        output_dir=created_skills_dir,
        model="gpt-4o" # Specify model
    )
    print(f"Created {len(created_paths)} skills:")
    for path in created_paths:
        print(f"- {path}")
except Exception as e:
    print(f"Creation failed (check API Key): {e}")

# Example 2: Create directly from Prompt
print("\n--- Creating skill from prompt ---")
try:
    created_paths_prompt = client.create(
        prompt="Create a tool that converts degrees Celsius to Fahrenheit",
        output_dir=created_skills_dir
    )
    for path in created_paths_prompt:
        print(f"- {path}")
except Exception as e:
    print(f"Creation failed: {e}")

--- Creating skill from trajectory ---
Created 1 skills:
- ./my_created_skills\file-format-renamer

--- Creating skill from prompt ---
- ./my_created_skills\celsius-to-fahrenheit-converter


## 4. Evaluate Skills (Evaluate)

Evaluate skill quality across multiple dimensions (Safety, Completeness, Executability, etc.).

In [23]:
# Evaluate the skill just downloaded or created
# Here we try to evaluate a skill in the download directory (if it exists)
import glob

# Find an existing skill path to evaluate
target_skill = None
if os.path.exists(download_dir) and os.listdir(download_dir):
    # Get the first subdirectory
    subdirs = [f.path for f in os.scandir(download_dir) if f.is_dir()]
    if subdirs:
        target_skill = subdirs[0]

if target_skill:
    print(f"--- Evaluating skill at: {target_skill} ---")
    try:
        # This step requires a valid API_KEY
        eval_result = client.evaluate(target=target_skill)
        print("Evaluation Result:")
        print(eval_result)
    except Exception as e:
        print(f"Evaluation failed (check API Key): {e}")
else:
    print("No local skill found to evaluate. Please download or create one first.")

--- Evaluating skill at: ./my_downloaded_skills\skill-creator ---
Evaluation Result:
{'safety': {'level': 'Good', 'reason': 'The skill focuses on creating modular, reusable components and explicitly avoids destructive or risky actions, with no dangerous tools or operations mentioned.'}, 'completeness': {'level': 'Good', 'reason': 'The SKILL.md provides detailed guidance on structuring skills, including workflows, resource types, and examples, ensuring clarity on how to create and use skills effectively.'}, 'executability': {'level': 'Good', 'reason': 'The skill is instruction-based and provides actionable steps and patterns for creating and organizing skills, which can be executed using typical LLM tools without ambiguity.'}, 'modifiability': {'level': 'Good', 'reason': 'The skill emphasizes modularity and progressive disclosure, making it easy to adapt or extend with additional resources or workflows as needed.'}, 'cost_awareness': {'level': 'Good', 'reason': 'The skill explicitly emp

## 5. Skill Relationship Analysis (Analyze)

Analyze relationships between multiple skills in a directory (e.g., dependency, composition, similarity).

In [24]:
# Analyze the directory of created skills
analyze_target_dir = created_skills_dir

if os.path.exists(analyze_target_dir) and len(os.listdir(analyze_target_dir)) > 0:
    print(f"--- Analyzing relationships in: {analyze_target_dir} ---")
    try:
        # This step requires a valid API_KEY, analyzes all skills in the directory
        relationships = client.analyze(skills_dir=analyze_target_dir)
        
        print("\nIdentified Relationships:")
        if relationships:
            for rel in relationships:
                print(f"{rel['source']} --[{rel['type']}]--> {rel['target']}")
        else:
            print("No significant relationships found between these skills.")
            
    except Exception as e:
        print(f"Analysis failed (check API Key): {e}")
else:
    print("No skills directory found to analyze.")

--- Analyzing relationships in: ./my_created_skills ---

Identified Relationships:
file-extension-renamer --[belong_to]--> file-renamer-tool
file-renamer-tool --[similar_to]--> file-extension-renamer
